<center>

# [Компьютерное зрение](http://rairi.ru/wiki/index.php/%D0%9A%D0%BE%D0%BC%D0%BF%D1%8C%D1%8E%D1%82%D0%B5%D1%80%D0%BD%D0%BE%D0%B5_%D0%B7%D1%80%D0%B5%D0%BD%D0%B8%D0%B5)

## <center> Семинар 12 - MinkowskiEngine

<a target="_blank" href="https://colab.research.google.com/github/alexmelekhin/cv_course_2023/blob/main/seminars/seminar_12/Seminar_12.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

***

# Установка MinkowskiEngine в Google Colab

При желании установить локально или где-либо еще - обращайтесь к инструкциям в репозитории: https://github.com/NVIDIA/MinkowskiEngine 

In [ ]:
!nvidia-smi

In [ ]:
!pip install torch==1.13.1 torchvision==0.14.1

In [ ]:
!pip install ninja

In [ ]:
import torch
print(f"torch version: {torch.__version__}")
print(f"Is CUDA available in torch?: {torch.cuda.is_available()}")

In [ ]:
!sudo apt-get install libopenblas-dev

In [ ]:
!pip install -U git+https://github.com/NVIDIA/MinkowskiEngine -v --no-deps \
                          --install-option="--force_cuda" \
                          --install-option="--blas=openblas"

In [ ]:
import torch
print(f"Is CUDA available in torch?: {torch.cuda.is_available()}")
import MinkowskiEngine as ME
print(f"Is CUDA available in MinkowskiEngine?: {ME.is_cuda_available()}")
ME.print_diagnostics()

!!! На Colab не запустилось из-за необходимости в древних версиях библиотек и nvidia toolkit, запустил локально через docker.

# ModelNet40-классификатор на MinkowskiEngine

Предлагается ознакомиться с оффициальными туториалами:

https://nvidia.github.io/MinkowskiEngine/demo/training.html

https://github.com/NVIDIA/MinkowskiEngine/blob/master/examples/training.py

https://nvidia.github.io/MinkowskiEngine/demo/modelnet40_classification.html


## Домашнее задание

Обучить простейшую модель классификации 3D объектов на датасете ModelNet40, воспользовавшись туториалом выше и кодом отсюда: https://github.com/NVIDIA/MinkowskiEngine/blob/master/examples/classification_modelnet40.py 

Вам предлагается обучить простейшую модель `minkfcnn` (аргумент `--network minkfcnn`).

В качестве отчета по заданию вам предлагается приложить результаты обучения (логи).

## Решение

Ниже минимально портированы нужные куски официального примера, без запуска внешнего скрипта.

Source: https://github.com/NVIDIA/MinkowskiEngine/blob/master/examples/classification_modelnet40.py

Source: https://github.com/NVIDIA/MinkowskiEngine/blob/master/examples/pointnet.py

In [ ]:
import glob
import os
import random
import urllib.request
import zipfile
from pathlib import Path
from types import SimpleNamespace

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import MinkowskiEngine as ME

config = SimpleNamespace(
    voxel_size=0.05,
    epochs=5,
    train_batches_per_epoch=250,
    val_batches=80,
    batch_size=4,
    lr=1e-3,
    weight_decay=1e-4,
    num_workers=2,
    stat_freq=25,
    weights="minkfcnn_modelnet40.pth",
    seed=777,
    translation=0.2,
    test_translation=0.0,
)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(config.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
def minkowski_collate_fn(list_data):
    coordinates_batch, features_batch, labels_batch = ME.utils.sparse_collate(
        [d["coordinates"] for d in list_data],
        [d["features"] for d in list_data],
        [d["label"] for d in list_data],
        dtype=torch.float32,
    )
    return {"coordinates": coordinates_batch, "features": features_batch, "labels": labels_batch}

class CoordinateTransformation:
    def __init__(self, scale_range=(0.9, 1.1), trans=0.25, jitter=0.025, clip=0.05):
        self.scale_range = scale_range
        self.trans = trans
        self.jitter = jitter
        self.clip = clip

    def __call__(self, coords):
        if random.random() < 0.9:
            coords *= np.random.uniform(self.scale_range[0], self.scale_range[1], size=[1, 3])
        if random.random() < 0.9:
            coords += np.random.uniform(-self.trans, self.trans, size=[1, 3])
        if random.random() < 0.7:
            coords += np.clip(self.jitter * (np.random.rand(len(coords), 3) - 0.5), -self.clip, self.clip)
        return coords

class CoordinateTranslation:
    def __init__(self, translation):
        self.trans = translation

    def __call__(self, coords):
        if self.trans > 0:
            coords += np.random.uniform(-self.trans, self.trans, size=[1, 3])
        return coords

def download_modelnet40_dataset():
    root = Path("modelnet40_ply_hdf5_2048")
    if root.exists():
        return
    zip_name = Path("modelnet40_ply_hdf5_2048.zip")
    urls = [
        "https://huggingface.co/datasets/Msun/modelnet40/resolve/main/modelnet40_ply_hdf5_2048.zip",
    ]
    if not zip_name.exists():
        last_error = None
        for url in urls:
            try:
                print("downloading", url)
                urllib.request.urlretrieve(url, zip_name)
                break
            except Exception as e:
                last_error = e
                if zip_name.exists():
                    zip_name.unlink()
        else:
            raise RuntimeError("Could not download ModelNet40") from last_error
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(".")

class ModelNet40H5(Dataset):
    def __init__(self, phase, data_root="modelnet40_ply_hdf5_2048", transform=None, num_points=2048):
        download_modelnet40_dataset()
        self.phase = "test" if phase in ["val", "test"] else "train"
        self.transform = transform
        self.num_points = num_points
        self.data, self.label = self.load_data(data_root, self.phase)

    def load_data(self, data_root, phase):
        data, labels = [], []
        for h5_name in glob.glob(os.path.join(data_root, f"ply_data_{phase}*.h5")):
            with h5py.File(h5_name) as f:
                data.extend(f["data"][:].astype("float32"))
                labels.extend(f["label"][:].astype("int64"))
        return np.stack(data), np.stack(labels)

    def __getitem__(self, i):
        xyz = self.data[i].copy()
        if self.phase == "train":
            np.random.shuffle(xyz)
        xyz = xyz[: self.num_points]
        if self.transform is not None:
            xyz = self.transform(xyz)
        xyz = torch.from_numpy(xyz).float()
        label = torch.from_numpy(self.label[i])
        return {"coordinates": xyz, "features": xyz, "label": label}

    def __len__(self):
        return len(self.data)

def make_data_loader(phase):
    is_train = phase == "train"
    transform = CoordinateTransformation(trans=config.translation) if is_train else CoordinateTranslation(config.test_translation)
    dataset = ModelNet40H5(phase=phase, transform=transform)
    return DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=is_train,
        num_workers=config.num_workers,
        collate_fn=minkowski_collate_fn,
    )

train_loader = make_data_loader("train")
val_loader = make_data_loader("val")
print(len(train_loader.dataset), len(val_loader.dataset))

In [ ]:
class MinkowskiFCNN(ME.MinkowskiNetwork):
    def __init__(self, in_channel, out_channel, embedding_channel=256, channels=(16, 24, 32, 48, 64), D=3):
        super().__init__(D)
        self.mlp1 = self.get_mlp_block(in_channel, channels[0])
        self.conv1 = self.get_conv_block(channels[0], channels[1], 3, 1)
        self.conv2 = self.get_conv_block(channels[1], channels[2], 3, 2)
        self.conv3 = self.get_conv_block(channels[2], channels[3], 3, 2)
        self.conv4 = self.get_conv_block(channels[3], channels[4], 3, 2)
        self.conv5 = nn.Sequential(
            self.get_conv_block(sum(channels[1:]), embedding_channel // 4, 3, 2),
            self.get_conv_block(embedding_channel // 4, embedding_channel // 2, 3, 2),
            self.get_conv_block(embedding_channel // 2, embedding_channel, 3, 2),
        )
        self.pool = ME.MinkowskiMaxPooling(kernel_size=3, stride=2, dimension=D)
        self.global_max_pool = ME.MinkowskiGlobalMaxPooling()
        self.global_avg_pool = ME.MinkowskiGlobalAvgPooling()
        self.final = nn.Sequential(
            self.get_mlp_block(embedding_channel * 2, 256),
            ME.MinkowskiDropout(),
            ME.MinkowskiLinear(256, out_channel, bias=True),
        )
        self.weight_initialization()

    def get_mlp_block(self, in_channel, out_channel):
        return nn.Sequential(
            ME.MinkowskiLinear(in_channel, out_channel, bias=False),
            ME.MinkowskiBatchNorm(out_channel),
            ME.MinkowskiLeakyReLU(),
        )

    def get_conv_block(self, in_channel, out_channel, kernel_size, stride):
        return nn.Sequential(
            ME.MinkowskiConvolution(in_channel, out_channel, kernel_size=kernel_size, stride=stride, dimension=self.D),
            ME.MinkowskiBatchNorm(out_channel),
            ME.MinkowskiLeakyReLU(),
        )

    def weight_initialization(self):
        for m in self.modules():
            if isinstance(m, ME.MinkowskiConvolution):
                ME.utils.kaiming_normal_(m.kernel, mode="fan_out", nonlinearity="relu")
            if isinstance(m, ME.MinkowskiBatchNorm):
                nn.init.constant_(m.bn.weight, 1)
                nn.init.constant_(m.bn.bias, 0)

    def forward(self, x):
        x = self.mlp1(x)
        y = self.conv1(x.sparse())
        y1 = self.pool(y)
        y2 = self.pool(self.conv2(y1))
        y3 = self.pool(self.conv3(y2))
        y4 = self.pool(self.conv4(y3))
        x = ME.cat(y1.slice(x), y2.slice(x), y3.slice(x), y4.slice(x))
        y = self.conv5(x.sparse())
        return self.final(ME.cat(self.global_max_pool(y), self.global_avg_pool(y))).F

net = MinkowskiFCNN(in_channel=3, out_channel=40).to(device)
net

In [ ]:
def create_input_batch(batch):
    coords = batch["coordinates"].clone()
    coords[:, 1:] = coords[:, 1:] / config.voxel_size
    return ME.TensorField(coordinates=coords, features=batch["features"], device=device)

def criterion(pred, labels, smoothing=True):
    labels = labels.contiguous().view(-1)
    if not smoothing:
        return F.cross_entropy(pred, labels)
    eps = 0.1
    n_class = pred.size(1)
    one_hot = torch.zeros_like(pred).scatter(1, labels.view(-1, 1), 1)
    one_hot = one_hot * (1 - eps) + (1 - one_hot) * eps / (n_class - 1)
    return -(one_hot * F.log_softmax(pred, dim=1)).sum(dim=1).mean()

def evaluate(loader, max_batches=None):
    net.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if max_batches is not None and i >= max_batches:
                break
            labels = batch["labels"].to(device).view(-1)
            logits = net(create_input_batch(batch))
            loss = criterion(logits, labels, smoothing=False)
            pred = logits.argmax(1)
            bs = len(labels)
            loss_sum += loss.item() * bs
            correct += (pred == labels).sum().item()
            total += bs
    net.train()
    return loss_sum / total, correct / total

optimizer = optim.AdamW(net.parameters(), lr=config.lr, weight_decay=config.weight_decay)
total_steps = config.epochs * config.train_batches_per_epoch
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
train_iter = iter(train_loader)
log_lines = []
best_acc = 0.0

net.train()
for epoch in range(1, config.epochs + 1):
    loss_sum = correct = total = 0
    for i in range(1, config.train_batches_per_epoch + 1):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        optimizer.zero_grad()
        labels = batch["labels"].to(device).view(-1)
        logits = net(create_input_batch(batch))
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 10.0)
        optimizer.step()
        scheduler.step()

        bs = len(labels)
        loss_sum += loss.item() * bs
        correct += (logits.argmax(1) == labels).sum().item()
        total += bs
        if i % config.stat_freq == 0:
            print(f"epoch {epoch}/{config.epochs} batch {i}/{config.train_batches_per_epoch} loss {loss.item():.3f}")

    train_loss = loss_sum / total
    train_acc = correct / total
    val_loss, val_acc = evaluate(val_loader, max_batches=config.val_batches)
    line = (
        f"Epoch {epoch}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )
    print(line)
    log_lines.append(line)
    if val_acc >= best_acc:
        best_acc = val_acc
        torch.save({"state_dict": net.state_dict(), "val_acc": val_acc, "epoch": epoch}, config.weights)
    torch.cuda.empty_cache()

Path("minkfcnn_modelnet40.log").write_text("\n".join(log_lines))
print(f"saved log and best weights, best_val_acc={best_acc:.4f}")